# Classification de la localisation tumorale — Random Forest

Ce notebook évalue un **Random Forest** pour prédire la localisation (`cort`, `dipg`, `midl`) à partir des blocs GE et CGH, avec une **nested cross-validation** pour une estimation non biaisée des performances.

## 0. Préparation

In [ ]:
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_classif
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score,
)
from sklearn.model_selection import (
    GridSearchCV, RepeatedStratifiedKFold, StratifiedKFold,
    permutation_test_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
root = Path.cwd()
DATA_DIR = root.parent / "data"

SEED = 42
LABEL_ORDER = ["cort", "dipg", "midl"]

# CV externe (évaluation non biaisée)
OUTER_CV = RepeatedStratifiedKFold(n_splits=4, n_repeats=5, random_state=SEED)
# CV interne (optimisation des hyperparamètres)
INNER_CV = StratifiedKFold(n_splits=4, shuffle=True, random_state=SEED)

# Grilles d'hyperparamètres
K_GE_GRID  = [40, 60, 80, 100, 120]
K_CGH_GRID = [20, 30, 40, 60]

RF_PARAM_GRID = {
    "clf__n_estimators": [100, 300, 500],
    "clf__max_depth": [3, 5, 8, None],
    "clf__min_samples_leaf": [1, 2, 3, 5],
    "clf__max_features": ["sqrt", "log2"],
}

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

## 1. Chargement des données

In [ ]:
def to_numeric_frame(df):
    return df.apply(lambda col: pd.to_numeric(col.astype(str).str.replace(",", ".", regex=False), errors="coerce"))


def load_block(block_name: str, split: str) -> pd.DataFrame:
    path = DATA_DIR / f"ge_cgh_locIGR__multiblocks__{block_name}__{split}.csv"
    df = pd.read_csv(path).rename(columns={"row_id": "patient_id"}).set_index("patient_id")
    return to_numeric_frame(df)


def load_targets(split: str) -> pd.Series:
    path = DATA_DIR / f"ge_cgh_locIGR__multiblocks__y__{split}.csv"
    y_df = pd.read_csv(path).rename(columns={"row_id": "patient_id"}).set_index("patient_id")
    y = y_df[LABEL_ORDER].idxmax(axis=1)
    y.name = "localisation"
    return y


def fill_missing_from_train(train_df, test_df):
    medians = train_df.median(axis=0)
    return train_df.fillna(medians), test_df.fillna(medians)


X_ge_train  = load_block("GE", "train")
X_ge_test   = load_block("GE", "test")
X_cgh_train = load_block("CGH", "train")
X_cgh_test  = load_block("CGH", "test")
y_train     = load_targets("train")
y_test      = load_targets("test")

train_ids = X_ge_train.index.intersection(X_cgh_train.index).intersection(y_train.index)
test_ids  = X_ge_test.index.intersection(X_cgh_test.index).intersection(y_test.index)

X_ge_train  = X_ge_train.loc[train_ids]
X_ge_test   = X_ge_test.loc[test_ids]
X_cgh_train = X_cgh_train.loc[train_ids]
X_cgh_test  = X_cgh_test.loc[test_ids]
y_train     = y_train.loc[train_ids]
y_test      = y_test.loc[test_ids]

X_ge_train, X_ge_test   = fill_missing_from_train(X_ge_train, X_ge_test)
X_cgh_train, X_cgh_test = fill_missing_from_train(X_cgh_train, X_cgh_test)

print(f"Train : {len(y_train)} échantillons  |  Test : {len(y_test)} échantillons")
print(f"GE : {X_ge_train.shape[1]} features  |  CGH : {X_cgh_train.shape[1]} features")
display(y_train.value_counts().reindex(LABEL_ORDER).to_frame("train").join(
    y_test.value_counts().reindex(LABEL_ORDER).to_frame("test")))

## 2. Fonctions utilitaires

In [ ]:
def prediction_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
    }


def bootstrap_ci(y_true, y_pred, metric_fn=balanced_accuracy_score, n_boot=2000, seed=SEED):
    rng = np.random.RandomState(seed)
    y_true_arr, y_pred_arr = np.asarray(y_true), np.asarray(y_pred)
    scores = [metric_fn(y_true_arr[idx], y_pred_arr[idx])
              for idx in (rng.choice(len(y_true_arr), len(y_true_arr), replace=True) for _ in range(n_boot))]
    lo, med, hi = np.percentile(scores, [2.5, 50, 97.5])
    return lo, med, hi


def confusion_table(y_true, y_pred):
    matrix = confusion_matrix(y_true, y_pred, labels=LABEL_ORDER)
    return pd.DataFrame(matrix,
                        index=[f"true_{l}" for l in LABEL_ORDER],
                        columns=[f"pred_{l}" for l in LABEL_ORDER])


def select_features(train_df, test_df, y_train, k):
    vt = VarianceThreshold()
    train_vt = vt.fit_transform(train_df)
    test_vt  = vt.transform(test_df)
    kept = train_df.columns[vt.get_support()]

    sel = SelectKBest(score_func=f_classif, k=min(k, len(kept)))
    train_sel = sel.fit_transform(train_vt, y_train)
    test_sel  = sel.transform(test_vt)
    selected  = kept[sel.get_support()]

    return (pd.DataFrame(train_sel, index=train_df.index, columns=selected),
            pd.DataFrame(test_sel,  index=test_df.index,  columns=selected),
            selected)

## 3. Nested Cross-Validation

```
┌─ Boucle externe (évaluation) ──────────────────────┐
│  Train outer          │  Val outer (score non biaisé)│
│  ┌─ Boucle interne ─┐ │                              │
│  │ Train │ Val       │ │                              │
│  │ inner │ inner     │ │                              │
│  │ (GridSearchCV)    │ │                              │
│  └──────────────────-┘ │                              │
└────────────────────────┴──────────────────────────────┘
```

In [ ]:
def nested_cv(pipeline, param_grid, X, y, outer_cv=OUTER_CV, inner_cv=INNER_CV):
    """
    Nested cross-validation.
    Retourne les scores de la boucle externe et les meilleurs paramètres de chaque fold.
    """
    outer_scores = []
    best_params_list = []
    feature_importances_list = []

    X_arr = np.asarray(X)

    for fold_i, (train_idx, val_idx) in enumerate(outer_cv.split(X_arr, y)):
        X_tr, X_va = X_arr[train_idx], X_arr[val_idx]
        y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]

        inner_search = GridSearchCV(
            pipeline, param_grid, cv=inner_cv,
            scoring="balanced_accuracy", refit=True, n_jobs=-1,
        )
        inner_search.fit(X_tr, y_tr)

        val_pred = inner_search.predict(X_va)
        score = balanced_accuracy_score(y_va, val_pred)
        outer_scores.append(score)
        best_params_list.append(inner_search.best_params_)

        # Sauvegarder les feature importances
        clf = inner_search.best_estimator_.named_steps["clf"]
        feature_importances_list.append(clf.feature_importances_)

    return np.array(outer_scores), best_params_list, feature_importances_list

## 4. Pipeline Random Forest

Le pipeline inclut :
1. `VarianceThreshold` pour éliminer les features constantes
2. `SelectKBest` pour réduire la dimensionnalité (le RF gère le bruit, mais 15 000 features avec 39 échantillons nécessitent un pré-filtre)
3. `RandomForestClassifier` avec `class_weight='balanced'`

In [ ]:
def make_rf_pipeline():
    return Pipeline([
        ("variance", VarianceThreshold()),
        ("select", SelectKBest(score_func=f_classif)),
        ("clf", RandomForestClassifier(
            class_weight="balanced",
            random_state=SEED,
            n_jobs=-1,
        )),
    ])

## 5. Bloc GE seul

In [ ]:
ge_pipe = make_rf_pipeline()
ge_grid = {"select__k": K_GE_GRID, **RF_PARAM_GRID}

print("Nested CV — Random Forest sur GE...")
ge_scores, ge_params, ge_fi = nested_cv(ge_pipe, ge_grid, X_ge_train, y_train)
print(f"  Balanced accuracy : {np.mean(ge_scores):.3f} ± {np.std(ge_scores):.3f}")

In [ ]:
# Refit sur tout le train
ge_final = GridSearchCV(make_rf_pipeline(), ge_grid,
                        cv=INNER_CV, scoring="balanced_accuracy", refit=True, n_jobs=-1)
ge_final.fit(X_ge_train, y_train)

ge_test_pred  = ge_final.predict(X_ge_test)
ge_train_pred = ge_final.predict(X_ge_train)

ge_test_metrics  = prediction_metrics(y_test, ge_test_pred)
ge_train_metrics = prediction_metrics(y_train, ge_train_pred)
ge_lo, ge_med, ge_hi = bootstrap_ci(y_test, ge_test_pred)

print(f"Best params : {ge_final.best_params_}")
print(f"CV interne  : {ge_final.best_score_:.3f}")
print(f"Train bal. acc. : {ge_train_metrics['balanced_accuracy']:.3f}")
print(f"Test  bal. acc. : {ge_test_metrics['balanced_accuracy']:.3f}  CI95=[{ge_lo:.3f}, {ge_hi:.3f}]")
print()
display(confusion_table(y_test, ge_test_pred))

## 6. Bloc CGH seul

In [ ]:
cgh_pipe = make_rf_pipeline()
cgh_grid = {"select__k": K_CGH_GRID, **RF_PARAM_GRID}

print("Nested CV — Random Forest sur CGH...")
cgh_scores, cgh_params, cgh_fi = nested_cv(cgh_pipe, cgh_grid, X_cgh_train, y_train)
print(f"  Balanced accuracy : {np.mean(cgh_scores):.3f} ± {np.std(cgh_scores):.3f}")

In [ ]:
cgh_final = GridSearchCV(make_rf_pipeline(), cgh_grid,
                         cv=INNER_CV, scoring="balanced_accuracy", refit=True, n_jobs=-1)
cgh_final.fit(X_cgh_train, y_train)

cgh_test_pred  = cgh_final.predict(X_cgh_test)
cgh_train_pred = cgh_final.predict(X_cgh_train)

cgh_test_metrics  = prediction_metrics(y_test, cgh_test_pred)
cgh_train_metrics = prediction_metrics(y_train, cgh_train_pred)
cgh_lo, cgh_med, cgh_hi = bootstrap_ci(y_test, cgh_test_pred)

print(f"Best params : {cgh_final.best_params_}")
print(f"CV interne  : {cgh_final.best_score_:.3f}")
print(f"Train bal. acc. : {cgh_train_metrics['balanced_accuracy']:.3f}")
print(f"Test  bal. acc. : {cgh_test_metrics['balanced_accuracy']:.3f}  CI95=[{cgh_lo:.3f}, {cgh_hi:.3f}]")
print()
display(confusion_table(y_test, cgh_test_pred))

## 7. Fusion précoce (GE + CGH concaténés)

In [ ]:
X_all_train = pd.concat([X_ge_train.add_prefix("GE__"), X_cgh_train.add_prefix("CGH__")], axis=1)
X_all_test  = pd.concat([X_ge_test.add_prefix("GE__"),  X_cgh_test.add_prefix("CGH__")],  axis=1)

early_pipe = make_rf_pipeline()
early_grid = {"select__k": [40, 60, 80, 100, 120, 150], **RF_PARAM_GRID}

print("Nested CV — Random Forest, fusion précoce...")
early_scores, early_params, early_fi = nested_cv(early_pipe, early_grid, X_all_train, y_train)
print(f"  Balanced accuracy : {np.mean(early_scores):.3f} ± {np.std(early_scores):.3f}")

In [ ]:
early_final = GridSearchCV(make_rf_pipeline(), early_grid,
                           cv=INNER_CV, scoring="balanced_accuracy", refit=True, n_jobs=-1)
early_final.fit(X_all_train, y_train)

early_test_pred  = early_final.predict(X_all_test)
early_train_pred = early_final.predict(X_all_train)

early_test_metrics  = prediction_metrics(y_test, early_test_pred)
early_train_metrics = prediction_metrics(y_train, early_train_pred)
early_lo, early_med, early_hi = bootstrap_ci(y_test, early_test_pred)

print(f"Best params : {early_final.best_params_}")
print(f"CV interne  : {early_final.best_score_:.3f}")
print(f"Train bal. acc. : {early_train_metrics['balanced_accuracy']:.3f}")
print(f"Test  bal. acc. : {early_test_metrics['balanced_accuracy']:.3f}  CI95=[{early_lo:.3f}, {early_hi:.3f}]")
print()
display(confusion_table(y_test, early_test_pred))

## 8. Fusion tardive (moyenne des probabilités GE + CGH)

On combine les `predict_proba` des Random Forests GE et CGH, pondérées par un coefficient `alpha` optimisé en CV.

In [ ]:
classes = ge_final.best_estimator_.named_steps["clf"].classes_

alpha_grid = np.arange(0.5, 1.01, 0.1)
best_alpha, best_alpha_score = 0.5, 0.0
skf = StratifiedKFold(n_splits=4, shuffle=True, random_state=SEED)

for alpha in alpha_grid:
    fold_scores = []
    for tr_idx, va_idx in skf.split(X_ge_train, y_train):
        p_ge  = ge_final.predict_proba(X_ge_train.iloc[va_idx])
        p_cgh = cgh_final.predict_proba(X_cgh_train.iloc[va_idx])
        p_blend = alpha * p_ge + (1 - alpha) * p_cgh
        preds = classes[np.argmax(p_blend, axis=1)]
        fold_scores.append(balanced_accuracy_score(y_train.iloc[va_idx], preds))
    mean_score = np.mean(fold_scores)
    if mean_score > best_alpha_score:
        best_alpha, best_alpha_score = alpha, mean_score

print(f"Meilleur alpha (poids GE) : {best_alpha:.2f}  (CV bal. acc. = {best_alpha_score:.3f})")

# Prédiction finale
late_test_proba  = best_alpha * ge_final.predict_proba(X_ge_test)  + (1 - best_alpha) * cgh_final.predict_proba(X_cgh_test)
late_train_proba = best_alpha * ge_final.predict_proba(X_ge_train) + (1 - best_alpha) * cgh_final.predict_proba(X_cgh_train)

late_test_pred  = classes[np.argmax(late_test_proba, axis=1)]
late_train_pred = classes[np.argmax(late_train_proba, axis=1)]

late_test_metrics  = prediction_metrics(y_test, late_test_pred)
late_train_metrics = prediction_metrics(y_train, late_train_pred)
late_lo, late_med, late_hi = bootstrap_ci(y_test, late_test_pred)

print(f"Train bal. acc. : {late_train_metrics['balanced_accuracy']:.3f}")
print(f"Test  bal. acc. : {late_test_metrics['balanced_accuracy']:.3f}  CI95=[{late_lo:.3f}, {late_hi:.3f}]")
print()
display(confusion_table(y_test, late_test_pred))

## 9. Tableau comparatif

In [ ]:
rows = []

def add_row(name, nested_scores, train_metrics, test_metrics, ci_lo, ci_hi):
    rows.append({
        "model": name,
        "nested_cv_bal_acc": f"{np.mean(nested_scores):.3f} ± {np.std(nested_scores):.3f}",
        "train_bal_acc": train_metrics["balanced_accuracy"],
        "test_bal_acc": test_metrics["balanced_accuracy"],
        "test_CI95": f"[{ci_lo:.3f}, {ci_hi:.3f}]",
        "test_macro_f1": test_metrics["macro_f1"],
        "train_test_gap": train_metrics["balanced_accuracy"] - test_metrics["balanced_accuracy"],
    })

add_row("GE seul (RF)",         ge_scores,    ge_train_metrics,    ge_test_metrics,    ge_lo,    ge_hi)
add_row("CGH seul (RF)",        cgh_scores,   cgh_train_metrics,   cgh_test_metrics,   cgh_lo,   cgh_hi)
add_row("Fusion précoce (RF)",  early_scores, early_train_metrics, early_test_metrics, early_lo, early_hi)
add_row("Fusion tardive (RF)",  ge_scores,    late_train_metrics,  late_test_metrics,  late_lo,  late_hi)

comparison = pd.DataFrame(rows).set_index("model")
comparison = comparison.sort_values("test_bal_acc", ascending=False)
display(comparison)

## 10. Distribution des scores nested CV

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

all_scores = {
    "GE (RF)": ge_scores,
    "CGH (RF)": cgh_scores,
    "Fusion précoce (RF)": early_scores,
}

positions = list(range(len(all_scores)))
bp = ax.boxplot(all_scores.values(), positions=positions, vert=True, patch_artist=True, widths=0.5)

colors = ["#4c72b0", "#dd8452", "#55a868"]
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_xticks(positions)
ax.set_xticklabels(all_scores.keys(), rotation=15, ha="right")
ax.set_ylabel("Balanced accuracy")
ax.set_title("Scores nested CV — Random Forest")
ax.axhline(y=1/3, color="red", linestyle="--", alpha=0.5, label="Hasard (1/3)")
ax.legend()
plt.tight_layout()
plt.show()

## 11. Test de permutation

In [ ]:
best_estimator = ge_final.best_estimator_
best_X = X_ge_train

score_obs, perm_scores, pvalue = permutation_test_score(
    best_estimator, best_X, y_train,
    scoring="balanced_accuracy",
    cv=StratifiedKFold(n_splits=4, shuffle=True, random_state=SEED),
    n_permutations=500,
    random_state=SEED,
    n_jobs=-1,
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(perm_scores, bins=30, density=True, alpha=0.7, color="steelblue", label="Permutations")
ax.axvline(score_obs, color="red", linewidth=2, label=f"Score observé = {score_obs:.3f}")
ax.set_xlabel("Balanced accuracy (CV)")
ax.set_ylabel("Densité")
ax.set_title(f"Test de permutation — Random Forest — p-value = {pvalue:.4f}")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Score observé : {score_obs:.3f}")
print(f"p-value :       {pvalue:.4f}")

## 12. Analyse des hyperparamètres sélectionnés (nested CV)

In [ ]:
params_df = pd.DataFrame(ge_params)

print(f"Distribution des hyperparamètres — GE (RF) — {len(ge_params)} folds externes")
for col in params_df.columns:
    print(f"\n  {col}:")
    print(f"    {params_df[col].value_counts().sort_index().to_dict()}")

## 13. Feature importances (Gini + stabilité across folds)

On agrège les feature importances du Random Forest à travers les folds de la nested CV pour identifier les features les plus stables et les plus discriminantes.

In [ ]:
# Refit pour avoir les noms de features
best_pipe = ge_final.best_estimator_
vt = best_pipe.named_steps["variance"]
sel = best_pipe.named_steps["select"]
clf = best_pipe.named_steps["clf"]

kept_cols = X_ge_train.columns[vt.get_support()]
selected_cols = kept_cols[sel.get_support()]

# Feature importances du modèle final
fi = pd.Series(clf.feature_importances_, index=selected_cols).sort_values(ascending=False)

top_n = 25
fig, ax = plt.subplots(figsize=(8, 7))
fi.head(top_n).plot.barh(ax=ax, color="steelblue")
ax.set_xlabel("Feature importance (Gini)")
ax.set_title(f"Top {top_n} features — Random Forest sur GE")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 14. Stabilité des feature importances à travers les folds

On regarde la variance des importances à travers les folds de la nested CV. Une importance stable (faible variance) indique un signal robuste.

In [ ]:
# Toutes les feature importances ont la même taille (= k sélectionné)
# Mais k peut varier d'un fold à l'autre, donc on utilise le modèle final
# et on fait un bootstrap des importances OOB

# Alternative : on refait la sélection + RF sur des bootstrap du train
from sklearn.utils import resample

n_bootstrap = 50
importance_matrix = np.zeros((n_bootstrap, len(selected_cols)))

for i in range(n_bootstrap):
    X_boot, y_boot = resample(X_ge_train, y_train, random_state=SEED + i, stratify=y_train)
    pipe_boot = make_rf_pipeline()
    pipe_boot.set_params(**ge_final.best_params_)
    pipe_boot.fit(X_boot, y_boot)
    
    # Récupérer les features sélectionnées dans ce bootstrap
    vt_b = pipe_boot.named_steps["variance"]
    sel_b = pipe_boot.named_steps["select"]
    kept_b = X_ge_train.columns[vt_b.get_support()]
    selected_b = kept_b[sel_b.get_support()]
    
    fi_b = pd.Series(pipe_boot.named_steps["clf"].feature_importances_, index=selected_b)
    # Aligner sur les colonnes du modèle final
    for j, col in enumerate(selected_cols):
        importance_matrix[i, j] = fi_b.get(col, 0.0)

fi_mean = importance_matrix.mean(axis=0)
fi_std  = importance_matrix.std(axis=0)

fi_stability = pd.DataFrame({
    "mean_importance": fi_mean,
    "std_importance": fi_std,
    "cv_importance": fi_std / (fi_mean + 1e-10),  # coefficient de variation
}, index=selected_cols).sort_values("mean_importance", ascending=False)

# Plot top features avec barres d'erreur
top = fi_stability.head(top_n)

fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(range(len(top)), top["mean_importance"], xerr=top["std_importance"],
        color="steelblue", alpha=0.8, capsize=3)
ax.set_yticks(range(len(top)))
ax.set_yticklabels(top.index, fontsize=8)
ax.set_xlabel("Feature importance (moyenne ± std sur 50 bootstraps)")
ax.set_title(f"Top {top_n} features — Stabilité des importances")
ax.invert_yaxis()
plt.tight_layout()
plt.show()